Cella 1: Setup, Import e Caricamento Dati

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Diciamo a Colab di spostarsi fisicamente nella cartella del tuo progetto
%cd /content/drive/MyDrive/DV-TM/

Mounted at /content/drive
/content/drive/MyDrive/DV-TM


In [2]:
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np

# Imposta il seed per la riproducibilità
torch.manual_seed(42)

# Path della cartella DATA (Modificabile dal prof)
DATA_PATH = './DATA/'

def load_data(filepath):
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

# Carichiamo i dati puliti
all_labeled_data = load_data(DATA_PATH + 'train_cleaned.jsonl')

# Splittiamo in Train (80%) e Validation (20%) per poter calcolare le metriche
train_data, val_data = train_test_split(all_labeled_data, test_size=0.2, random_state=42)

print(f"Frasi di Training: {len(train_data)}")
print(f"Frasi di Validation: {len(val_data)}")

Frasi di Training: 2513
Frasi di Validation: 629


cella 2: creazione dei vocabolari

In [3]:
word2idx = {"<PAD>": 0, "<UNK>": 1}
tag2idx = {"<PAD>": 0}

for sentence in train_data:
    for word in sentence['tokens']:
        if word not in word2idx:
            word2idx[word] = len(word2idx)
    for tag in sentence['labels']:
        if tag not in tag2idx:
            tag2idx[tag] = len(tag2idx)

idx2tag = {v: k for k, v in tag2idx.items()}

print(f"Dimensione Vocabolario Parole: {len(word2idx)}")
print(f"Tags unici trovati: {list(tag2idx.keys())}")

Dimensione Vocabolario Parole: 103
Tags unici trovati: ['<PAD>', 'B-JOBTITLE', 'I-JOBTITLE', 'O', 'B-COMPANY', 'I-COMPANY', 'B-SKILL', 'I-SKILL', 'B-LOCATION', 'I-LOCATION']


Cella 3: Dataset e DataLoader
Gestiamo le frasi convertendole in tensori e allineando le lunghezze con il padding.

In [4]:
class JobNERDataset(Dataset):
    def __init__(self, data, word2idx, tag2idx, max_len=30):
        self.data = data
        self.word2idx = word2idx
        self.tag2idx = tag2idx
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        tokens = item['tokens']
        labels = item.get('labels', ['O'] * len(tokens))

        # Converti parole in indici (usa <UNK> se non esiste nel vocab)
        word_indices = [self.word2idx.get(w, self.word2idx["<UNK>"]) for w in tokens]
        tag_indices = [self.tag2idx[t] for t in labels]

        # Troncamento o Padding
        if len(word_indices) > self.max_len:
            word_indices = word_indices[:self.max_len]
            tag_indices = tag_indices[:self.max_len]
        else:
            pad_len = self.max_len - len(word_indices)
            word_indices = word_indices + [self.word2idx["<PAD>"]] * pad_len
            tag_indices = tag_indices + [self.tag2idx["<PAD>"]] * pad_len

        return torch.tensor(word_indices), torch.tensor(tag_indices)

BATCH_SIZE = 32
train_loader = DataLoader(JobNERDataset(train_data, word2idx, tag2idx), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(JobNERDataset(val_data, word2idx, tag2idx), batch_size=BATCH_SIZE)

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report

class LSTM_NER(nn.Module):
    def __init__(self, vocab_size, tagset_size, embedding_dim=128, hidden_dim=256):
        super(LSTM_NER, self).__init__()
        # 1. Layer di Embedding
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        # 2. LSTM Unidirezionale (bidirectional=False)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim,
                            num_layers=1, bidirectional=False, batch_first=True)

        # 3. Layer Lineare per la classificazione finale
        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)

    def forward(self, x):
        embeds = self.embedding(x)
        lstm_out, _ = self.lstm(embeds)
        tag_space = self.hidden2tag(lstm_out)
        return tag_space

# Inizializziamo il modello
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_lstm = LSTM_NER(len(word2idx), len(tag2idx)).to(device)

# Loss function (ignoriamo il padding) e Ottimizzatore
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model_lstm.parameters(), lr=0.001)

print(f"Modello LSTM (Unidirezionale) caricato su: {device}")

Modello LSTM (Unidirezionale) caricato su: cuda


In [13]:
EPOCHS = 10

print("Inizio Addestramento LSTM Unidirezionale...")
for epoch in range(EPOCHS):
    model_lstm.train()
    total_loss = 0
    for sentences, tags in train_loader:
        sentences, tags = sentences.to(device), tags.to(device)

        optimizer.zero_grad()
        predictions = model_lstm(sentences)

        # Reshape per calcolare l'errore: appiattiamo le dimensioni
        predictions = predictions.view(-1, predictions.shape[-1])
        tags = tags.view(-1)

        loss = criterion(predictions, tags)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoca {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")

# --- VALUTAZIONE SUL VALIDATION SET ---
model_lstm.eval()
all_preds = []
all_true = []

with torch.no_grad():
    for sentences, tags in val_loader:
        sentences, tags = sentences.to(device), tags.to(device)
        predictions = model_lstm(sentences)

        # Estraiamo la classe con la probabilità più alta
        preds = torch.argmax(predictions, dim=-1)

        preds = preds.view(-1).cpu().numpy()
        tags = tags.view(-1).cpu().numpy()

        # Escludiamo i token di <PAD> (indice 0) dal calcolo delle metriche
        mask = tags != 0
        all_preds.extend(preds[mask])
        all_true.extend(tags[mask])

# Mappiamo gli indici numerici di nuovo alle etichette originali (BIO)
true_labels = [idx2tag[idx] for idx in all_true]
pred_labels = [idx2tag[idx] for idx in all_preds]

print("\n--- RISULTATI DELLA VALUTAZIONE (LSTM UNIDIREZIONALE) ---")
print(classification_report(true_labels, pred_labels, zero_division=0))

Inizio Addestramento LSTM Unidirezionale...
Epoca 1/10 | Loss: 0.6237
Epoca 2/10 | Loss: 0.0117
Epoca 3/10 | Loss: 0.0036
Epoca 4/10 | Loss: 0.0018
Epoca 5/10 | Loss: 0.0011
Epoca 6/10 | Loss: 0.0008
Epoca 7/10 | Loss: 0.0006
Epoca 8/10 | Loss: 0.0004
Epoca 9/10 | Loss: 0.0003
Epoca 10/10 | Loss: 0.0003

--- RISULTATI DELLA VALUTAZIONE (LSTM UNIDIREZIONALE) ---
              precision    recall  f1-score   support

   B-COMPANY       1.00      1.00      1.00       629
  B-JOBTITLE       1.00      1.00      1.00       629
  B-LOCATION       1.00      1.00      1.00       629
     B-SKILL       1.00      1.00      1.00       629
   I-COMPANY       1.00      1.00      1.00       239
  I-JOBTITLE       1.00      1.00      1.00       629
  I-LOCATION       1.00      1.00      1.00       197
     I-SKILL       1.00      1.00      1.00       262
           O       1.00      1.00      1.00      5324

    accuracy                           1.00      9167
   macro avg       1.00      1.00      1

Modello BiLSTM vanilla

In [5]:
class StandardBiLSTM_NER(nn.Module):
    def __init__(self, vocab_size, tagset_size, embedding_dim=128, hidden_dim=256):
        super(StandardBiLSTM_NER, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        # BiLSTM classica e "corposa"
        self.lstm = nn.LSTM(embedding_dim, hidden_dim // 2,
                            num_layers=1, bidirectional=True, batch_first=True)

        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)

    def forward(self, x):
        # Nessun dropout, passiamo direttamente gli embedding
        embeds = self.embedding(x)
        lstm_out, _ = self.lstm(embeds)
        tag_space = self.hidden2tag(lstm_out)
        return tag_space

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = StandardBiLSTM_NER(len(word2idx), len(tag2idx)).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)
# Learning rate standard
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"Modello BASELINE istanziato e caricato su: {device}")

Modello BASELINE istanziato e caricato su: cuda


In [11]:
EPOCHS = 10

print("Inizio Addestramento...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for sentences, tags in train_loader:
        sentences, tags = sentences.to(device), tags.to(device)

        optimizer.zero_grad()
        predictions = model(sentences)

        # Reshape per la loss function: (batch_size * max_len, num_classes)
        predictions = predictions.view(-1, predictions.shape[-1])
        tags = tags.view(-1)

        loss = criterion(predictions, tags)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoca {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")

# --- VALUTAZIONE (METRICHE) ---
model.eval()
all_preds = []
all_true = []

with torch.no_grad():
    for sentences, tags in val_loader:
        sentences, tags = sentences.to(device), tags.to(device)
        predictions = model(sentences)

        # Prendi l'indice con probabilità maggiore
        preds = torch.argmax(predictions, dim=-1)

        # Appiattiamo i tensori e rimuoviamo il padding per calcolare le metriche reali
        preds = preds.view(-1).cpu().numpy()
        tags = tags.view(-1).cpu().numpy()

        # Filtriamo via i token di <PAD> (indice 0)
        mask = tags != 0
        all_preds.extend(preds[mask])
        all_true.extend(tags[mask])

# Convertiamo gli indici numerici in label testuali (B-COMPANY, O, ecc.)
true_labels = [idx2tag[idx] for idx in all_true]
pred_labels = [idx2tag[idx] for idx in all_preds]

print("\n--- RISULTATI DELLA VALUTAZIONE ---")
print(classification_report(true_labels, pred_labels, zero_division=0))

Inizio Addestramento...
Epoca 1/10 | Loss: 0.0001
Epoca 2/10 | Loss: 0.0001
Epoca 3/10 | Loss: 0.0001
Epoca 4/10 | Loss: 0.0001
Epoca 5/10 | Loss: 0.0001
Epoca 6/10 | Loss: 0.0001
Epoca 7/10 | Loss: 0.0001
Epoca 8/10 | Loss: 0.0001
Epoca 9/10 | Loss: 0.0001
Epoca 10/10 | Loss: 0.0001

--- RISULTATI DELLA VALUTAZIONE ---
              precision    recall  f1-score   support

   B-COMPANY       1.00      1.00      1.00       629
  B-JOBTITLE       1.00      1.00      1.00       629
  B-LOCATION       1.00      1.00      1.00       629
     B-SKILL       1.00      1.00      1.00       629
   I-COMPANY       1.00      1.00      1.00       239
  I-JOBTITLE       1.00      1.00      1.00       629
  I-LOCATION       1.00      1.00      1.00       197
     I-SKILL       1.00      1.00      1.00       262
           O       1.00      1.00      1.00      5324

    accuracy                           1.00      9167
   macro avg       1.00      1.00      1.00      9167
weighted avg       1.00     

Se dovessimo vedere che il modello va in overfitting, allora potremmo provare a modificarlo e renderlo più leggero.

Il Modello BiLSTM "Light"

In [7]:
class LightBiLSTM_NER(nn.Module):
    def __init__(self, vocab_size, tagset_size, embedding_dim=64, hidden_dim=64):
        super(LightBiLSTM_NER, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim // 2,
                            num_layers=1, bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(0.3) # Aiuta contro l'overfitting
        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)

    def forward(self, x):
        embeds = self.dropout(self.embedding(x))
        lstm_out, _ = self.lstm(embeds)
        tag_space = self.hidden2tag(lstm_out)
        return tag_space

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LightBiLSTM_NER(len(word2idx), len(tag2idx)).to(device)

# Loss function che ignora il <PAD> per non falsare l'errore
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=0.002)

print(f"Modello istanziato e caricato su: {device}")

Modello istanziato e caricato su: cuda


Cella 5: Training Loop ed Estrazione delle Metriche

In [8]:
EPOCHS = 10

print("Inizio Addestramento...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for sentences, tags in train_loader:
        sentences, tags = sentences.to(device), tags.to(device)

        optimizer.zero_grad()
        predictions = model(sentences)

        # Reshape per la loss function: (batch_size * max_len, num_classes)
        predictions = predictions.view(-1, predictions.shape[-1])
        tags = tags.view(-1)

        loss = criterion(predictions, tags)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoca {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")

# --- VALUTAZIONE (METRICHE) ---
model.eval()
all_preds = []
all_true = []

with torch.no_grad():
    for sentences, tags in val_loader:
        sentences, tags = sentences.to(device), tags.to(device)
        predictions = model(sentences)

        # Prendi l'indice con probabilità maggiore
        preds = torch.argmax(predictions, dim=-1)

        # Appiattiamo i tensori e rimuoviamo il padding per calcolare le metriche reali
        preds = preds.view(-1).cpu().numpy()
        tags = tags.view(-1).cpu().numpy()

        # Filtriamo via i token di <PAD> (indice 0)
        mask = tags != 0
        all_preds.extend(preds[mask])
        all_true.extend(tags[mask])

# Convertiamo gli indici numerici in label testuali (B-COMPANY, O, ecc.)
true_labels = [idx2tag[idx] for idx in all_true]
pred_labels = [idx2tag[idx] for idx in all_preds]

print("\n--- RISULTATI DELLA VALUTAZIONE ---")
print(classification_report(true_labels, pred_labels, zero_division=0))

Inizio Addestramento...
Epoca 1/10 | Loss: 1.1349
Epoca 2/10 | Loss: 0.1074
Epoca 3/10 | Loss: 0.0207
Epoca 4/10 | Loss: 0.0095
Epoca 5/10 | Loss: 0.0058
Epoca 6/10 | Loss: 0.0038
Epoca 7/10 | Loss: 0.0028
Epoca 8/10 | Loss: 0.0022
Epoca 9/10 | Loss: 0.0016
Epoca 10/10 | Loss: 0.0013

--- RISULTATI DELLA VALUTAZIONE ---
              precision    recall  f1-score   support

   B-COMPANY       1.00      1.00      1.00       629
  B-JOBTITLE       1.00      1.00      1.00       629
  B-LOCATION       1.00      1.00      1.00       629
     B-SKILL       1.00      1.00      1.00       629
   I-COMPANY       1.00      1.00      1.00       239
  I-JOBTITLE       1.00      1.00      1.00       629
  I-LOCATION       1.00      1.00      1.00       197
     I-SKILL       1.00      1.00      1.00       262
           O       1.00      1.00      1.00      5324

    accuracy                           1.00      9167
   macro avg       1.00      1.00      1.00      9167
weighted avg       1.00     

In [14]:
import torch

# Carichiamo il test set vero (quello re-tokenizzato che hai sistemato nella EDA)
# Usiamo il DATA_PATH che abbiamo definito all'inizio
test_data_real = load_data(DATA_PATH + 'test_retokenized.jsonl')

# Prendiamo le prime 5 frasi
sample_sentences = test_data_real[:5]

# Mettiamo il modello in modalità valutazione
model_lstm.eval()

print("🔍 PROVA DEL NOVE: Inferenza sul Test Set")
print("=" * 50)

with torch.no_grad():
    for i, item in enumerate(sample_sentences):
        tokens = item['tokens']

        # Convertiamo le parole in indici (usando <UNK> se la parola è nuova)
        word_indices = [word2idx.get(w, word2idx["<UNK>"]) for w in tokens]

        # Creiamo il tensore (aggiungendo la dimensione del batch) e lo mandiamo alla GPU
        x = torch.tensor([word_indices]).to(device)

        # Facciamo la previsione
        predictions = model_lstm(x)

        # Prendiamo la classe con la probabilità più alta
        preds = torch.argmax(predictions, dim=-1)[0].cpu().numpy()

        # Mappiamo gli indici alle etichette testuali
        pred_labels = [idx2tag[idx] for idx in preds]

        # Stampiamo il risultato allineato in due colonne
        print(f"FRASE {i+1}: {' '.join(tokens)}")
        print("-" * 30)
        for token, label in zip(tokens, pred_labels):
            # Stampiamo solo le entità trovate (ignoriamo le "O" per una lettura più pulita)
            if label != "O":
                print(f"{token:15} -> {label}")
        print("=" * 50)

🔍 PROVA DEL NOVE: Inferenza sul Test Set
FRASE 1: Product Manager position available at Global Solutions . Requirements : SQL . Location : Seattle .
------------------------------
Product         -> B-JOBTITLE
Manager         -> I-JOBTITLE
Global          -> B-COMPANY
Solutions       -> I-COMPANY
SQL             -> B-SKILL
Seattle         -> B-LOCATION
FRASE 2: Excellent opportunity : Business Analyst at CloudServices , Boston . Skills : leadership .
------------------------------
Business        -> B-JOBTITLE
Analyst         -> I-JOBTITLE
CloudServices   -> B-COMPANY
Boston          -> B-LOCATION
leadership      -> B-SKILL
FRASE 3: Global Solutions in Denver needs Data Scientist proficient in Python .
------------------------------
Global          -> B-COMPANY
Solutions       -> I-COMPANY
Denver          -> B-LOCATION
Data            -> B-JOBTITLE
Scientist       -> I-JOBTITLE
Python          -> B-SKILL
FRASE 4: Seeking Data Scientist at NextGen Industries located in Austin . Must hav

In [15]:
def stress_test(model, sentence):
    model.eval()
    # Dividiamo la frase in token (molto semplice)
    tokens = sentence.split()

    # Convertiamo in indici usando il vocabolario creato nel training
    # Se una parola non esiste, usiamo <UNK>
    word_indices = [word2idx.get(w, word2idx["<UNK>"]) for w in tokens]

    # Trasformiamo in tensore e aggiungiamo la dimensione del batch (1)
    x = torch.tensor([word_indices]).to(device)

    with torch.no_grad():
        predictions = model(x)
        preds = torch.argmax(predictions, dim=-1)[0].cpu().numpy()

    # Mappiamo i risultati
    pred_labels = [idx2tag[idx] for idx in preds]

    print(f"STRESS TEST: {sentence}")
    print("-" * 30)
    found = False
    for token, label in zip(tokens, pred_labels):
        if label != "O":
            print(f"{token:15} -> {label}")
            found = True
    if not found:
        print("Nessuna entità trovata (il modello è confuso!)")

# --- PROVA QUI LE TUE FRASI ---

# Esempio 1: Frase con entità conosciute ma struttura strana
stress_test(model_lstm, "Yesterday I saw that Apple is looking for a Barman in Naples for Excel tasks")

# Esempio 2: Frase molto colloquiale
stress_test(model_lstm, "I am a Python ninja and I want to work for Google in Rome")

STRESS TEST: Yesterday I saw that Apple is looking for a Barman in Naples for Excel tasks
------------------------------
Yesterday       -> B-LOCATION
I               -> B-LOCATION
saw             -> B-LOCATION
that            -> B-LOCATION
Apple           -> B-LOCATION
Barman          -> I-JOBTITLE
Naples          -> B-LOCATION
Excel           -> B-SKILL
tasks           -> I-SKILL
STRESS TEST: I am a Python ninja and I want to work for Google in Rome
------------------------------
I               -> B-LOCATION
am              -> B-LOCATION
Python          -> B-SKILL
ninja           -> I-JOBTITLE
and             -> I-JOBTITLE
I               -> B-LOCATION
want            -> B-LOCATION
work            -> I-JOBTITLE
Google          -> B-SKILL
Rome            -> B-LOCATION
